## Classification and Logistic Regression

*Note: The code in this notebook was generated with the assistance of Claude.*

I will be adding to this notebook throughout the week! 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, precision_score, recall_score, RocCurveDisplay

In [ ]:
# This dataset can be used to predict whether a patient has a 10-year risk of future CHD (coronary heart disease)
fram = pd.read_csv("../data/framingham.csv")
fram.head()

In [ ]:
# Do we have any missing data? 
fram.isna().sum()

In [ ]:
# For simplicity, we'll drop rows with any missing values (about 14% of rows)
# We will come back to this dataset to look more at imputation
# For now, we are going to just use rows that have all the data
fram = fram.dropna()
fram.shape

In [ ]:
# What fraction of people developed coronary heart disease (CHD) within 10 years?
fram['TenYearCHD'].value_counts(normalize=True)

In [ ]:
# We want to predict whether or not someone develops CHD within 10 years (1 or 0) based on a variety of risk factors
y = fram['TenYearCHD']

# There are three different types of variables in this dataset:
# - education is categorical (levels 1-4), so we one-hot encode it
# - binary variables are already 0/1, so we leave them alone
# - numeric variables
cats = ["education"]
bins = ["male", "currentSmoker", "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes"]
nums = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]

# Note: For this first pass, we going to use
# All of the numeric veriables
# Only a single categorical variable (Male or female)
male = ["male"]
X = fram[male + nums]

In [ ]:
# Let's build a logistic regression model
# Note: We are NOT going to scale anything for this first go around, to demonstrate why scaling is important
# This is problematic and in practice, we would scale everything first. 

logreg = LogisticRegression().fit(X,y)

In [ ]:
# We'll come back to the warning later. 
# Let's look at predicted vs actual CHD, and look at the confusion matrix

p = logreg.predict_proba(X)
#p = logreg.predict(X)
p

In [ ]:
# Note that this returns probability estimates for 0 or 1, and if we add each pair it should sum to 1. 
# We can look at just the second column (prediction of CHD) and get information about both

p = logreg.predict_proba(X)[:, 1] # second column
y_hat = (p >=0.5).astype(int) # if p >= 0.5, set to 1 (we can vary this threshold!)

In [ ]:
# Now let's create a dataframe of actual vs predicted
results = pd.DataFrame(
    {
    "Actual_CHD": y,
    "Predicted_prob_CHD": p.round(3),
    "Predicted_label": y_hat
})

results

In [ ]:
# Now we look at the confusion matrix, which gives us the following information:
# [[True Negative, False Positive],[False Negative, True Positive]]

confusion_matrix(y, y_hat)

In [ ]:
# We can put this in a slightly more digestible form
tn, fp, fn, tp = confusion_matrix(y, y_hat).ravel().tolist()
tn, fp, fn, tp

In [ ]:
# We can also plot the confusion matrix to visualize this. 
cm = confusion_matrix(y, y_hat, labels=logreg.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
# The results above show a lot of false negatives (e.g. the true label is 1, but the predicted label is 0)
fn_results = results[(results["Actual_CHD"] == 1) & (results["Predicted_label"] == 0)]

print(fn_results["Predicted_prob_CHD"].describe())

plt.hist(fn_results["Predicted_prob_CHD"], bins=20, edgecolor="black")
plt.axvline(0.5, color="red", linestyle="--", label="Threshold = 0.5")
plt.xlabel("Predicted probability of CHD")
plt.ylabel("Count")
plt.title(f"False Negatives (n = {len(fn_results)})")
plt.legend()
plt.show()

In [ ]:

# What happens to our confusion matrix if we lower the threshold to 0.2?
y_hat_alt = (p >=0.2).astype(int)
confusion_matrix(y, y_hat_alt)

In [ ]:
# We can also normalize
confusion_matrix(y, y_hat_alt, normalize="true")

In [ ]:
# What does changing the threshold tell us about the tradeoff between false negatives and false positives? 
cm = confusion_matrix(y, y_hat_alt, labels=logreg.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues")
plt.show()

In the example above, we are trading off false negatives for false positives. Let's see if scaling the numeric variables gives us a better answer. 

In [ ]:
# First, let's look at the scale of the numeric variables.
fram.describe()


In [ ]:
## There is a fair bit of variability in the numeric variables so let's scale them
# We are going to look at two different preprocessors:
# First, we will use the standard scaler only on the numeric variables, passing through the categorical
# Second, we will use the standard scaler on ALL variables

# Remember that Logistic regression does regularization by default, which depends on scaling, even for categorical variables

# Option 1: pass "male" through unchanged (it's already 0/1), scale only the numeric variables
preprocess_pass = ColumnTransformer(transformers=[
    ("binary", "passthrough", male),
    ("numeric", StandardScaler(), nums)
])

# Option 2: scale everything, including the 0/1 "male" variable
preprocess_all = ColumnTransformer(transformers=[
    ("all", StandardScaler(), male + nums)
])

pipe_pass = Pipeline(steps=[
    ("preprocess", preprocess_pass),
    ("model", LogisticRegression())  
])

pipe_all = Pipeline(steps=[
    ("preprocess", preprocess_all),
    ("model", LogisticRegression())  
])

# Note that we are using the X and y that we defined above
pipe_pass.fit(X, y)
pipe_all.fit(X, y)


In [ ]:
# Now let's compare the confusion matrices for these side by side

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, pipe, title in zip(axes,
                           [pipe_pass, pipe_all],
                           ["Passthrough male, scale numeric", "Scale everything"]):
    y_hat_pipe = pipe.predict(X)
    cm = confusion_matrix(y, y_hat_pipe, labels=pipe.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe.classes_)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# What if we adjust the threshold to be 0.2 instead of 0.5? 
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, pipe, title in zip(axes,
                           [pipe_pass, pipe_all],
                           ["Passthrough male, scale numeric", "Scale everything"]):
    y_hat_pipe = (pipe.predict_proba(X)[:, 1] >= 0.2).astype(int)
    cm = confusion_matrix(y, y_hat_pipe, labels=pipe.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe.classes_)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
for name, pipe in [("pass", pipe_pass), ("all", pipe_all)]:
    model = pipe.named_steps["model"]
    feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
    coefs = pd.Series(model.coef_.ravel(), index=feat_names)
    print(f"--- {name} --- intercept: {model.intercept_[0]:.4f}")
    print(coefs)

In [ ]:
# Now, let's look at accuracy/precision/recall and the auc score  
# We will define a function so we can do this repeatedly

def get_metrics(y_true, p, threshold=0.5):
    y_pred = (p >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, p),  # AUC uses probabilities, not labels
    }

p_in = pipe_all.predict_proba(X)[:, 1]
get_metrics(y, p_in)

In [ ]:
# We can also plot the ROC curve
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y, p_in, name="In-sample", ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random guess (AUC = 0.5)")
ax.set_title("ROC curve: in-sample")
ax.legend()
plt.show()


In [ ]:
# Now let's repeat all of this using cross-validation
# NOTE: We need to use Stratified KFold here in order to preserve the class balance in each set 

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
p_cv = cross_val_predict(pipe_all, X, y, cv=cv, method="predict_proba")[:, 1]

y_hat_cv = (p_cv >= 0.5).astype(int)
cm = confusion_matrix(y, y_hat_cv)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot(cmap="Blues")
plt.title("Out-of-sample (5-fold CV)")
plt.show()

In [ ]:
# We can also look at mean accuracy/precision/recall/and AUC from the cross validation
# Recall for linear regression we looked at RMSE like this:
# score = -cross_val_score(pipe, X, y, cv=cv, n_jobs=-1, scoring="neg_root_mean_squared_error").mean()
# Here we are using cross_validate instead because it allows us to calculate multiple metrics at once

scoring_metrics = ["accuracy", "precision", "recall", "roc_auc"]

# Run multi-metric cross-validation
cv_results = cross_validate(
    pipe_all, X, y, cv=cv, scoring=scoring_metrics, return_train_score=False
)

# Print the mean performance across all folds
print(f"Mean Accuracy:  {cv_results['test_accuracy'].mean():.4f}")
print(f"Mean Precision: {cv_results['test_precision'].mean():.4f}")
print(f"Mean Recall:    {cv_results['test_recall'].mean():.4f}")
print(f"Mean ROC-AUC:   {cv_results['test_roc_auc'].mean():.4f}")

In [ ]:
# ROC Curve
cv_results = cross_validate(pipe_all, X, y, cv=cv,
                            return_estimator=True, return_indices=True)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_cv_results(cv_results, X, y, ax=ax, plot_chance_level=True)
ax.set_title("ROC curve: out-of-sample (5-fold CV)")
plt.show()
